# 04. エンドツーエンド

`run_hypothesis()` を呼ぶ。**Web アプリのワーカーが呼ぶのと同じ関数**なので、
ここで通ればそのまま API 経由でも通る。

```
backend/app/worker.py  ─┐
                        ├─> biomni_hypo.pipeline.run_hypothesis()
notebooks/04 (ここ)     ─┘
```

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
print("repo root:", ROOT)

In [ ]:
from biomni_hypo.config import Settings, apply_biomni_env, install_hint, missing_dependencies
from biomni_hypo.models import apply_model_selection, list_local_models
from biomni_hypo.policy import ResourcePolicy

missing = missing_dependencies()
assert not missing, f"依存が足りません。実行してください: {install_hint(missing)}"

settings = Settings()
settings.max_steps = 30
settings.max_hypotheses = 5
policy = ResourcePolicy.load(settings.policy_path)

catalog = list_local_models(settings, policy)
print(catalog.as_table())

# 使いたいモデルを指定する（None ならローカルから既定を選ぶ）
CHOICE = None
_catalog, notes = apply_model_selection(settings, policy, model=CHOICE, catalog=catalog)
for note in notes:
    print("⚠️ ", note)

# モデルが確定したあとに呼ぶこと（BIOMNI_LLM に選択したモデル名が入る / §4.3）
apply_biomni_env(settings)

print(f"\n使用モデル: {settings.model} / num_ctx {settings.num_ctx:,} / policy v{policy.version}")

## エージェントを 1 度だけ構築して使い回す

`build_agent()` は重い（ツールレジストリ構築とノウハウ文書ロード）。
ノートブック内では 1 回作って `bundle` を渡し回す。

In [ ]:
from biomni_hypo.agent_factory import build_agent

bundle = build_agent(settings, policy)
print(bundle.report)

## ラン実行

`on_event` は Web アプリでは SSE に流れるコールバック。ここでは進捗表示に使う。

In [ ]:
from biomni_hypo.pipeline import run_hypothesis, summarize
from biomni_hypo.question import QuestionMode, ResearchQuestion

# 調べたいことを構造化して入力する。text 以外は任意だが、埋めるほど探索が安定する
question = ResearchQuestion(
    text="トリプルネガティブ乳がんで PARP 阻害剤耐性を規定する因子の候補は？",
    mode=QuestionMode.HYPOTHESIS,
    organism="ヒト",
    context="トリプルネガティブ乳がん、オラパリブ投与下",
    focus=["BRCA1", "BRCA2", "相同組換え修復"],
    background="BRCA 変異型では奏効するが、非変異型で耐性例が報告されている。",
    max_hypotheses=5,
)

# 入力の質を先に確認する（error があると run_hypothesis が実行前に落とす）
for hint in question.hints():
    print({"error": "❌", "warning": "⚠️ ", "info": "ℹ️ "}[hint.severity.value], hint.message)

print("\n--- エージェントに渡すプロンプト ---")
print(question.to_prompt(settings.prompt_language))

プロンプトを見てから実行する。何を投げたか分からないまま結果だけ出てくる状態を作らない。
組み立てたプロンプトは `RunResult.prompt` にも残り、レポートにも載る。

In [ ]:
def on_event(kind, payload):
    if kind == "phase":
        print(f"\n── {payload['phase']} ──")
    elif kind == "step":
        head = (payload.get("code") or payload.get("text") or "").strip().replace("\n", " ")[:80]
        print(f"[{payload['idx']:2d}] {payload['kind']:15s} {head}")
    elif kind in ("input_hints", "verification", "done"):
        print(kind, payload)

result = run_hypothesis(question, settings=settings, policy=policy, bundle=bundle, on_event=on_event)
print()
print(summarize(result))

## 品質チェック（受け入れ基準）

docs/design/08-roadmap.md §8.2 / §8.3 の指標をここで測る。
モデルやプロンプトを変えたら、このセルの数字を比べる。

In [ ]:
v = result.verification
checks = {
    "AC-1 observation の自己生成なし": result.extra.get("hallucinated_observations", 0) == 0,
    "AC-3 全仮説に根拠がある": all(h.evidence for h in result.hypotheses),
    "AC-4 検証失敗の引用が仮説に残っていない":
        all(ev.verification_status.value != "failed" for h in result.hypotheses for ev in h.evidence),
    "引用検証率 >= 0.95": v.rate >= 0.95,
    "根拠付き仮説率 >= 0.8":
        (len(result.hypotheses) / max(1, len(result.hypotheses) + len(result.unsupported_ideas))) >= 0.8,
}
for name, ok in checks.items():
    print(("✅" if ok else "❌"), name)

print()
print("引用検証率      :", f"{v.rate:.0%}", v.model_dump())
print("仮説            :", len(result.hypotheses), "件（未裏付け", len(result.unsupported_ideas), "件）")
print("破棄した未知 eid:", result.extra.get("unknown_eids", []))
print("所要            :", result.extra.get("duration_sec"), "秒")

## 仮説を読む

In [ ]:
for i, h in enumerate(result.hypotheses, 1):
    print(f"\n{'=' * 70}\n仮説 {i}. {h.statement}")
    print(f"確度 {h.confidence} / 新規性 {h.novelty}")
    print(f"根拠:")
    for ev in h.evidence:
        print(f"  [{ev.verification_status.value:14s}] {ev.kind.value:11s} {ev.identifier}")
        print(f"      {ev.why}")
        print(f"      抜粋: {ev.excerpt[:100]}")
        print(f"      由来: ステップ {ev.step_idx}")
    print(f"検証: {h.test_plan.experiment} / {h.test_plan.readout}")

## レポートを保存

In [ ]:
import pathlib

from biomni_hypo.report import to_markdown

out_dir = pathlib.Path(settings.workspace_path) / "reports"
out_dir.mkdir(parents=True, exist_ok=True)

md_path = out_dir / f"{result.id}.md"
json_path = out_dir / f"{result.id}.json"
md_path.write_text(to_markdown(result), encoding="utf-8")
json_path.write_text(result.model_dump_json(indent=2), encoding="utf-8")
print("保存:", md_path)
print("保存:", json_path)

In [ ]:
from IPython.display import Markdown, display
display(Markdown(to_markdown(result, include_trace=False)))

## Web アプリ側で同じことをする

```bash
uvicorn backend.app.main:app --reload --port 8000
```

```bash
curl -s -X POST localhost:8000/api/runs \
  -H 'content-type: application/json' \
  -d '{"question": "...", "model": "qwen3:14b"}'

curl -N localhost:8000/api/runs/<run_id>/events      # SSE でトレースが流れる
curl -s localhost:8000/api/runs/<run_id>/report      # Markdown レポート
```

ワーカーは `run_hypothesis()` をラン 1 本ごとの子プロセスで呼ぶ
（LLM が生成したコードを API サーバと同じプロセスで実行しないため）。